# SOL Option Data Multi-Venue Collector

This notebook builds a normalized option dataset for SOL pairs (USD/USDT/USDC where available) from centralized and decentralized venues.

It is designed for **pricing-quality data collection**, not a specific pricing model.

In [1]:
import os
import math
import time
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

NOW_UTC = datetime.now(timezone.utc)
MAX_DTE_DAYS = 45
MIN_DTE_DAYS = 1

OPTION_COLS = [
    'venue', 'symbol', 'underlying', 'quote_ccy', 'option_type',
    'strike', 'expiry_utc', 'T_years', 'spot',
    'mark', 'bid', 'ask', 'iv', 'oi', 'volume',
    'raw_timestamp_utc', 'fetch_timestamp_utc'
]

print('Setup complete.')

Setup complete.


In [2]:
def safe_get(url, params=None, timeout=12):
    try:
        r = requests.get(url, params=params, timeout=timeout)
        if r.status_code != 200:
            return None
        return r.json()
    except Exception:
        return None


def yfrac(expiry_dt, now_dt):
    return max(0.0, (expiry_dt - now_dt).total_seconds() / (365.0 * 24.0 * 3600.0))


def within_dte(T):
    d = T * 365.0
    return (MIN_DTE_DAYS <= d <= MAX_DTE_DAYS)


def parse_bybit_symbol(sym):
    # SOL-26APR26-150-P
    p = sym.split('-')
    if len(p) != 4:
        return None
    try:
        expiry = datetime.strptime(p[1], '%d%b%y').replace(tzinfo=timezone.utc)
        strike = float(p[2])
        cp = p[3]
        if cp not in ('C', 'P'):
            return None
        return expiry, strike, cp
    except Exception:
        return None


def parse_deribit_symbol(sym):
    # SOL-26APR26-150-C
    p = sym.split('-')
    if len(p) != 4:
        return None
    try:
        expiry = datetime.strptime(p[1], '%d%b%y').replace(tzinfo=timezone.utc)
        strike = float(p[2])
        cp = p[3]
        if cp not in ('C', 'P'):
            return None
        return expiry, strike, cp
    except Exception:
        return None


def normalize_row(**kwargs):
    row = {k: np.nan for k in OPTION_COLS}
    row.update(kwargs)
    return row

In [3]:
def fetch_bybit_options_sol(now_utc):
    rows = []
    spot_js = safe_get('https://api.bybit.com/v5/market/tickers', {'category': 'linear', 'symbol': 'SOLUSDT'})
    chain_js = safe_get('https://api.bybit.com/v5/market/tickers', {'category': 'option', 'baseCoin': 'SOL'})
    if not spot_js or not chain_js:
        return rows
    try:
        spot = float(spot_js['result']['list'][0]['lastPrice'])
    except Exception:
        return rows

    for t in chain_js.get('result', {}).get('list', []):
        sym = t.get('symbol', '')
        parsed = parse_bybit_symbol(sym)
        if not parsed:
            continue
        expiry, strike, cp = parsed
        T = yfrac(expiry, now_utc)
        if not within_dte(T):
            continue
        try:
            mark = float(t.get('markPrice') or 0.0)
            bid = float(t.get('bid1Price') or 0.0)
            ask = float(t.get('ask1Price') or 0.0)
            iv = float(t.get('markIv') or 0.0)
            oi = float(t.get('openInterest') or 0.0)
            vol = float(t.get('volume24h') or 0.0)
            if mark <= 0 or iv <= 0:
                continue
        except Exception:
            continue

        rows.append(normalize_row(
            venue='bybit', symbol=sym, underlying='SOL', quote_ccy='USDT',
            option_type='call' if cp == 'C' else 'put',
            strike=strike, expiry_utc=expiry.isoformat(), T_years=T, spot=spot,
            mark=mark, bid=bid, ask=ask, iv=iv, oi=oi, volume=vol,
            raw_timestamp_utc=np.nan, fetch_timestamp_utc=now_utc.isoformat()
        ))
    return rows


def fetch_binance_options_sol(now_utc):
    rows = []
    exch = safe_get('https://eapi.binance.com/eapi/v1/exchangeInfo')
    if not exch:
        return rows
    for s in exch.get('optionSymbols', []):
        if s.get('underlying') != 'SOLUSDT':
            continue
        sym = s.get('symbol', '')
        side = s.get('side', '')
        if side not in ('CALL', 'PUT'):
            continue
        try:
            strike = float(s.get('strikePrice'))
            exp_dt = datetime.fromtimestamp(int(s.get('expiryDate')) / 1000, tz=timezone.utc)
            T = yfrac(exp_dt, now_utc)
            if not within_dte(T):
                continue
        except Exception:
            continue

        tk = safe_get('https://eapi.binance.com/eapi/v1/ticker', {'symbol': sym})
        if not tk or not isinstance(tk, list) or len(tk) == 0:
            continue
        rec = tk[0]
        try:
            spot = float(rec.get('underlyingPrice') or 0.0)
            mark = float(rec.get('markPrice') or rec.get('lastPrice') or 0.0)
            bid = float(rec.get('bidPrice') or 0.0)
            ask = float(rec.get('askPrice') or 0.0)
            iv = float(rec.get('markIV') or 0.0)
            oi = float(rec.get('openInterest') or 0.0)
            vol = float(rec.get('volume') or 0.0)
            if mark <= 0 or iv <= 0 or spot <= 0:
                continue
        except Exception:
            continue

        rows.append(normalize_row(
            venue='binance', symbol=sym, underlying='SOL', quote_ccy='USDT',
            option_type='call' if side == 'CALL' else 'put',
            strike=strike, expiry_utc=exp_dt.isoformat(), T_years=T, spot=spot,
            mark=mark, bid=bid, ask=ask, iv=iv, oi=oi, volume=vol,
            raw_timestamp_utc=np.nan, fetch_timestamp_utc=now_utc.isoformat()
        ))
    return rows


def fetch_deribit_options_sol(now_utc):
    rows = []
    inst = safe_get('https://www.deribit.com/api/v2/public/get_instruments', {
        'currency': 'SOL', 'kind': 'option', 'expired': 'false'
    })
    if not inst:
        return rows
    for it in inst.get('result', []):
        name = it.get('instrument_name', '')
        parsed = parse_deribit_symbol(name)
        if not parsed:
            continue
        expiry, strike, cp = parsed
        T = yfrac(expiry, now_utc)
        if not within_dte(T):
            continue

        bk = safe_get('https://www.deribit.com/api/v2/public/get_book_summary_by_instrument', {
            'instrument_name': name
        })
        if not bk or not bk.get('result'):
            continue
        rec = bk['result'][0]
        try:
            spot = float(rec.get('underlying_price') or 0.0)
            iv = float(rec.get('mark_iv') or 0.0) / 100.0
            mark_u = float(rec.get('mark_price') or 0.0)
            bid_u = float(rec.get('bid_price') or 0.0)
            ask_u = float(rec.get('ask_price') or 0.0)
            mark = mark_u * spot
            bid = bid_u * spot if bid_u > 0 else 0.0
            ask = ask_u * spot if ask_u > 0 else 0.0
            oi = float(rec.get('open_interest') or 0.0)
            vol = float(rec.get('volume_usd') or 0.0)
            if mark <= 0 or iv <= 0 or spot <= 0:
                continue
        except Exception:
            continue

        rows.append(normalize_row(
            venue='deribit', symbol=name, underlying='SOL', quote_ccy='USD',
            option_type='call' if cp == 'C' else 'put',
            strike=strike, expiry_utc=expiry.isoformat(), T_years=T, spot=spot,
            mark=mark, bid=bid, ask=ask, iv=iv, oi=oi, volume=vol,
            raw_timestamp_utc=np.nan, fetch_timestamp_utc=now_utc.isoformat()
        ))
    return rows


def fetch_okx_options_sol(now_utc):
    rows = []
    inst = safe_get('https://www.okx.com/api/v5/public/instruments', {
        'instType': 'OPTION', 'uly': 'SOL-USD'
    })
    if not inst:
        return rows
    for it in inst.get('data', []):
        try:
            inst_id = it['instId']
            cp = it['optType']  # C/P
            strike = float(it['stk'])
            exp_ms = int(it['expTime'])
            expiry = datetime.fromtimestamp(exp_ms / 1000, tz=timezone.utc)
            T = yfrac(expiry, now_utc)
            if cp not in ('C', 'P') or not within_dte(T):
                continue
        except Exception:
            continue

        tk = safe_get('https://www.okx.com/api/v5/market/ticker', {'instId': inst_id})
        if not tk or not tk.get('data'):
            continue
        rec = tk['data'][0]
        try:
            mark = float(rec.get('last') or 0.0)
            bid = float(rec.get('bidPx') or 0.0)
            ask = float(rec.get('askPx') or 0.0)
            vol = float(rec.get('volCcy24h') or 0.0)
            if mark <= 0:
                continue
        except Exception:
            continue

        # OKX ticker endpoint does not always expose IV directly in this path.
        rows.append(normalize_row(
            venue='okx', symbol=inst_id, underlying='SOL', quote_ccy='USD',
            option_type='call' if cp == 'C' else 'put',
            strike=strike, expiry_utc=expiry.isoformat(), T_years=T, spot=np.nan,
            mark=mark, bid=bid, ask=ask, iv=np.nan, oi=np.nan, volume=vol,
            raw_timestamp_utc=np.nan, fetch_timestamp_utc=now_utc.isoformat()
        ))
    return rows


CONNECTORS = {
    'bybit': fetch_bybit_options_sol,
    'binance': fetch_binance_options_sol,
    'deribit': fetch_deribit_options_sol,
    'okx': fetch_okx_options_sol,
}

print('Connectors loaded:', ', '.join(CONNECTORS.keys()))

Connectors loaded: bybit, binance, deribit, okx


In [4]:
# Lyra/Derive connector: currently returns live SOL option quotes in this environment.
def fetch_lyra_options_sol(now_utc):
    rows = []

    inst = safe_get('https://api.lyra.finance/public/get_instruments', {
        'currency': 'SOL',
        'instrument_type': 'option',
        'expired': 'false',
    })
    if not inst:
        return rows

    instruments = inst.get('result', [])
    if not instruments:
        return rows

    # Group active option instruments by expiry date (YYYYMMDD expected by get_tickers).
    expiry_dates = set()
    for it in instruments:
        if not it.get('is_active', False):
            continue
        name = it.get('instrument_name', '')
        # Example: SOL-20260501-100-C
        parts = name.split('-')
        if len(parts) == 4 and parts[0] == 'SOL':
            expiry_dates.add(parts[1])

    for exp in sorted(expiry_dates):
        tk = safe_get('https://api.lyra.finance/public/get_tickers', {
            'currency': 'SOL',
            'instrument_type': 'option',
            'expiry_date': exp,
        })
        if not tk:
            continue

        tickers = tk.get('result', {}).get('tickers', {})
        for sym, rec in tickers.items():
            parts = sym.split('-')
            if len(parts) != 4:
                continue
            try:
                expiry = datetime.strptime(parts[1], '%Y%m%d').replace(tzinfo=timezone.utc)
                strike = float(parts[2])
                cp = parts[3]
                if cp not in ('C', 'P'):
                    continue
                T = yfrac(expiry, now_utc)
                if not within_dte(T):
                    continue

                # Lyra fields are mostly strings.
                mark = float(rec.get('M') or 0.0)
                bid = float(rec.get('B') or 0.0)
                ask = float(rec.get('A') or 0.0)
                spot = float(rec.get('I') or np.nan)
                iv = float((rec.get('option_pricing') or {}).get('i') or np.nan)
                stats = rec.get('stats') or {}
                oi = float(stats.get('oi') or 0.0)
                vol = float(stats.get('v') or 0.0)
                ts = rec.get('t')
                if mark <= 0 or not np.isfinite(spot):
                    continue
            except Exception:
                continue

            raw_ts = datetime.fromtimestamp(ts / 1000, tz=timezone.utc).isoformat() if ts else np.nan
            rows.append(normalize_row(
                venue='lyra', symbol=sym, underlying='SOL', quote_ccy='USD',
                option_type='call' if cp == 'C' else 'put',
                strike=strike, expiry_utc=expiry.isoformat(), T_years=T, spot=spot,
                mark=mark, bid=bid, ask=ask, iv=iv, oi=oi, volume=vol,
                raw_timestamp_utc=raw_ts, fetch_timestamp_utc=now_utc.isoformat(),
            ))

    return rows

# Ensure Lyra is included in the connector map.
CONNECTORS['lyra'] = fetch_lyra_options_sol
print('Added connector: lyra')

Added connector: lyra


In [5]:
def fetch_spot_reference():
    spots = {}
    # CEX references
    b = safe_get('https://api.bybit.com/v5/market/tickers', {'category': 'linear', 'symbol': 'SOLUSDT'})
    if b and b.get('result', {}).get('list'):
        try:
            spots['bybit_SOLUSDT'] = float(b['result']['list'][0]['lastPrice'])
        except Exception:
            pass

    bn = safe_get('https://api.binance.com/api/v3/ticker/price', {'symbol': 'SOLUSDT'})
    if bn and 'price' in bn:
        try:
            spots['binance_SOLUSDT'] = float(bn['price'])
        except Exception:
            pass

    okx = safe_get('https://www.okx.com/api/v5/market/ticker', {'instId': 'SOL-USDT'})
    if okx and okx.get('data'):
        try:
            spots['okx_SOLUSDT'] = float(okx['data'][0]['last'])
        except Exception:
            pass

    cb = safe_get('https://api.coinbase.com/v2/prices/SOL-USD/spot')
    if cb and cb.get('data', {}).get('amount'):
        try:
            spots['coinbase_SOLUSD'] = float(cb['data']['amount'])
        except Exception:
            pass

    # DEX reference via Birdeye (SOL mint)
    # If unavailable, it is skipped.
    be = safe_get('https://public-api.birdeye.so/defi/price', {'address': 'So11111111111111111111111111111111111111112'})
    if be and be.get('data', {}).get('value') is not None:
        try:
            spots['birdeye_SOLUSD'] = float(be['data']['value'])
        except Exception:
            pass

    if len(spots) == 0:
        return spots, np.nan
    consensus = float(np.median(list(spots.values())))
    return spots, consensus


def fetch_realized_vol_birdeye(days=45):
    # Public endpoint may be rate-limited without API key in some environments.
    now_ts = int(time.time())
    t0 = now_ts - days * 86400
    js = safe_get('https://public-api.birdeye.so/defi/ohlcv', {
        'address': 'So11111111111111111111111111111111111111112',
        'type': '1D',
        'time_from': t0,
        'time_to': now_ts,
    })
    if not js:
        return np.nan, np.nan
    items = js.get('data', {}).get('items', [])
    closes = []
    for c in items:
        try:
            closes.append(float(c['c']))
        except Exception:
            pass
    closes = np.array([x for x in closes if x > 0], dtype=float)
    if len(closes) < 10:
        return np.nan, np.nan
    lr = np.diff(np.log(closes))
    rv_all = float(np.std(lr, ddof=1) * np.sqrt(365))
    rv_7 = float(np.std(lr[-7:], ddof=1) * np.sqrt(365)) if len(lr) >= 7 else np.nan
    return rv_all, rv_7


spot_map, spot_consensus = fetch_spot_reference()
rv_30_proxy, rv_7_proxy = fetch_realized_vol_birdeye(60)
print('Spot references:', spot_map)
print('Spot consensus:', spot_consensus)
print('RV proxies (annualized):', {'rv_30_proxy': rv_30_proxy, 'rv_7_proxy': rv_7_proxy})

Spot references: {'bybit_SOLUSDT': 86.2, 'binance_SOLUSDT': 86.22, 'okx_SOLUSDT': 86.23, 'coinbase_SOLUSD': 86.25}
Spot consensus: 86.225
RV proxies (annualized): {'rv_30_proxy': nan, 'rv_7_proxy': nan}


In [6]:
all_rows = []
status = {}
for name, fn in CONNECTORS.items():
    t0 = time.time()
    try:
        rows = fn(NOW_UTC)
        status[name] = {'ok': True, 'n': len(rows), 'sec': round(time.time() - t0, 2)}
        all_rows.extend(rows)
    except Exception as e:
        status[name] = {'ok': False, 'n': 0, 'sec': round(time.time() - t0, 2), 'err': str(e)}

df = pd.DataFrame(all_rows, columns=OPTION_COLS)

if len(df) == 0:
    print('No option rows fetched. Status:', status)
else:
    # Fill missing spot with consensus when unavailable.
    if not np.isnan(spot_consensus):
        df['spot'] = df['spot'].fillna(spot_consensus)

    # Basic quality checks
    df['mid'] = np.where((df['bid'] > 0) & (df['ask'] > 0), 0.5 * (df['bid'] + df['ask']), np.nan)
    df['ba_spread_pct'] = np.where(df['mid'] > 0, (df['ask'] - df['bid']) / df['mid'] * 100.0, np.nan)
    df['intrinsic_put'] = np.where(df['option_type'] == 'put', np.maximum(df['strike'] - df['spot'], 0.0), np.nan)
    df['intrinsic_call'] = np.where(df['option_type'] == 'call', np.maximum(df['spot'] - df['strike'], 0.0), np.nan)
    df['below_intrinsic_flag'] = np.where(
        df['option_type'] == 'put',
        df['mark'] + 1e-9 < df['intrinsic_put'],
        df['mark'] + 1e-9 < df['intrinsic_call']
    )

    # Export
    ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    out_csv = os.path.join(DATA_DIR, f'sol_options_multivenue_{ts}.csv')
    out_status = os.path.join(DATA_DIR, f'sol_options_multivenue_status_{ts}.json')
    df.to_csv(out_csv, index=False)
    with open(out_status, 'w', encoding='utf-8') as f:
        json.dump(status, f, indent=2)

    print('Fetch status by venue:')
    print(pd.DataFrame(status).T)
    print('\nRows:', len(df), '| Venues:', sorted(df['venue'].dropna().unique().tolist()))
    print('Missing IV %:', round(100 * df['iv'].isna().mean(), 2))
    print('Below intrinsic count:', int(df['below_intrinsic_flag'].sum()))
    print('Median bid-ask spread %:', float(np.nanmedian(df['ba_spread_pct'])))
    print('\nSaved:', out_csv)
    print('Saved:', out_status)

    display(df.head(20))

Fetch status by venue:
           ok    n    sec
bybit    True    0   0.92
binance  True    0  25.17
deribit  True    0   0.14
okx      True    0   0.32
lyra     True  218   7.65

Rows: 218 | Venues: ['lyra']
Missing IV %: 0.0
Below intrinsic count: 35
Median bid-ask spread %: 0.0

Saved: data/sol_options_multivenue_20260414_091910.csv
Saved: data/sol_options_multivenue_status_20260414_091910.json


,venue,symbol,underlying,quote_ccy,option_type,strike,expiry_utc,T_years,spot,mark,...,iv,oi,volume,raw_timestamp_utc,fetch_timestamp_utc,mid,ba_spread_pct,intrinsic_put,intrinsic_call,below_intrinsic_flag
0,lyra,SOL-20260416-89-P,SOL,USD,put,89.0,2026-04-16T00:00:00+00:00,0.004417,86.2,3.2,...,0.58336,0.0,0.0,2026-04-14T09:19:04.154000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,2.8,NaN,False
1,lyra,SOL-20260416-79-C,SOL,USD,call,79.0,2026-04-16T00:00:00+00:00,0.004417,86.2,7.3,...,0.65488,0.0,0.0,2026-04-14T09:19:04.010000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,NaN,7.2,False
2,lyra,SOL-20260416-81-C,SOL,USD,call,81.0,2026-04-16T00:00:00+00:00,0.004417,86.2,5.4,...,0.62473,0.0,0.0,2026-04-14T09:19:04.819000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,NaN,5.2,False
3,lyra,SOL-20260416-95-P,SOL,USD,put,95.0,2026-04-16T00:00:00+00:00,0.004417,86.2,8.7,...,0.64699,0.0,0.0,2026-04-14T09:19:05.019000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,8.8,NaN,True
4,lyra,SOL-20260416-81-P,SOL,USD,put,81.0,2026-04-16T00:00:00+00:00,0.004417,86.2,0.1,...,0.62473,0.0,0.0,2026-04-14T09:19:04.207000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,0.0,NaN,False
5,lyra,SOL-20260416-86-P,SOL,USD,put,86.0,2026-04-16T00:00:00+00:00,0.004417,86.2,1.2,...,0.55421,0.0,0.0,2026-04-14T09:19:04.717000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,0.0,NaN,False
6,lyra,SOL-20260416-76-C,SOL,USD,call,76.0,2026-04-16T00:00:00+00:00,0.004417,86.2,10.2,...,0.69911,0.0,0.0,2026-04-14T09:19:04.010000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,NaN,10.2,False
7,lyra,SOL-20260416-75-C,SOL,USD,call,75.0,2026-04-16T00:00:00+00:00,0.004417,86.2,11.2,...,0.71362,0.0,0.0,2026-04-14T09:19:04.415000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,NaN,11.2,False
8,lyra,SOL-20260416-89-C,SOL,USD,call,89.0,2026-04-16T00:00:00+00:00,0.004417,86.2,0.5,...,0.58336,0.0,0.0,2026-04-14T09:19:03.909000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,NaN,0.0,False
9,lyra,SOL-20260416-78-C,SOL,USD,call,78.0,2026-04-16T00:00:00+00:00,0.004417,86.2,8.3,...,0.66975,0.0,0.0,2026-04-14T09:19:04.010000+00:00,2026-04-14T09:18:33.671240+00:00,NaN,NaN,NaN,8.2,False


## Calibration: Simulation-Ready Option Parameters

This section derives robust, real-data parameters for simulation from the normalized option dataset.

Outputs include:
- ATM IV anchor (7d-equivalent)
- IV skew vs moneyness
- Term slope vs expiry
- Tradable spread proxy and liquidity penalty
- Stress bounds (floor/cap)

These are exported as JSON for direct use in simulation notebooks.

In [7]:
# Derive robust option-simulation parameters from fetched data.
if 'df' not in globals() or len(df) == 0:
    print('No option rows available; cannot calibrate parameters.')
else:
    d = df.copy()

    # Base filters: valid numeric values needed for calibration.
    d = d[np.isfinite(d['spot']) & np.isfinite(d['strike']) & np.isfinite(d['T_years']) & np.isfinite(d['iv'])]
    d = d[(d['spot'] > 0) & (d['strike'] > 0) & (d['T_years'] > 0) & (d['iv'] > 0)]

    if len(d) == 0:
        print('All rows filtered out; cannot calibrate parameters.')
    else:
        d['moneyness'] = d['strike'] / d['spot']
        d['dte'] = d['T_years'] * 365.0
        d['log_mny'] = np.log(np.clip(d['moneyness'], 1e-6, None))
        d['weight'] = 1.0 + np.clip(pd.to_numeric(d['volume'], errors='coerce').fillna(0.0), 0.0, None)

        # Tradable subset (prefer rows with positive bid/ask, but allow fallback).
        tradable = d[(pd.to_numeric(d['bid'], errors='coerce').fillna(0.0) > 0) &
                     (pd.to_numeric(d['ask'], errors='coerce').fillna(0.0) > 0)]
        used = tradable if len(tradable) >= max(20, int(0.1 * len(d))) else d

        # ATM-near subset for IV anchor.
        atm = used[(used['moneyness'] >= 0.97) & (used['moneyness'] <= 1.03)]
        if len(atm) < 10:
            atm = used[(used['moneyness'] >= 0.94) & (used['moneyness'] <= 1.06)]

        # 7d equivalent IV anchor (if enough term spread, fit vs dte, else median ATM).
        iv_anchor = float(np.median(atm['iv'])) if len(atm) > 0 else float(np.median(used['iv']))

        # Weighted linear model for IV surface: iv = b0 + b1*log_mny + b2*T
        X = np.column_stack([
            np.ones(len(used)),
            used['log_mny'].to_numpy(dtype=float),
            used['T_years'].to_numpy(dtype=float),
        ])
        y = used['iv'].to_numpy(dtype=float)
        w = np.sqrt(np.clip(used['weight'].to_numpy(dtype=float), 1.0, None))

        Xw = X * w[:, None]
        yw = y * w
        try:
            beta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
            b0, b_mny, b_t = [float(x) for x in beta]
        except Exception:
            b0, b_mny, b_t = float(iv_anchor), 0.0, 0.0

        # Convert slopes to interpretable units.
        # Approx IV change for 10% OTM move (K/S from 1.00 to 0.90 => log change ~ ln(0.9)).
        skew_per_10pct = float(b_mny * np.log(0.9))
        term_per_30d = float(b_t * (30.0 / 365.0))

        # Bid/ask spread proxy from available tradable rows.
        if len(tradable) > 0:
            mid = 0.5 * (tradable['bid'].astype(float) + tradable['ask'].astype(float))
            spread_pct = (tradable['ask'].astype(float) - tradable['bid'].astype(float)) / np.clip(mid, 1e-9, None)
            spread_median = float(np.nanmedian(spread_pct))
            spread_p75 = float(np.nanpercentile(spread_pct, 75))
        else:
            # Fallback: use conservative defaults when quoted books are absent.
            spread_median = 0.10
            spread_p75 = 0.20

        # IV robust bounds for simulation stress control.
        iv_floor = float(np.nanpercentile(used['iv'], 10))
        iv_cap = float(np.nanpercentile(used['iv'], 90))

        # IV/RV diagnostics if RV proxy available.
        rv = rv_30_proxy if 'rv_30_proxy' in globals() else np.nan
        iv_rv_median = float(np.median(used['iv']) / rv) if np.isfinite(rv) and rv > 0 else np.nan

        # Liquidity penalty: raises effective spread when sparse tradable quotes.
        tradable_ratio = float(len(tradable) / max(len(d), 1))
        liquidity_penalty = float(np.clip(1.0 + (0.6 - tradable_ratio), 1.0, 1.8))

        calibrated = {
            'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'sample': {
                'rows_total': int(len(df)),
                'rows_used': int(len(used)),
                'rows_tradable': int(len(tradable)),
                'venues': sorted(list(set(df['venue'].dropna().astype(str).tolist()))),
            },
            'option_pricing_params': {
                'iv_atm_anchor': float(iv_anchor),
                'iv_floor': float(iv_floor),
                'iv_cap': float(iv_cap),
                'iv_skew_per_10pct_moneyness': float(skew_per_10pct),
                'iv_term_slope_per_30d': float(term_per_30d),
                'spread_pct_median': float(spread_median),
                'spread_pct_stress_p75': float(spread_p75),
                'liquidity_penalty_multiplier': float(liquidity_penalty),
                'iv_rv_median': float(iv_rv_median) if np.isfinite(iv_rv_median) else None,
            },
            'notes': [
                'Use iv_atm_anchor as baseline implied vol in BS pricing.',
                'Adjust iv by skew and term: iv_eff = clamp(iv_anchor + skew_term + term_term, iv_floor, iv_cap).',
                'Apply spread_pct_median (or stress p75) as execution cost uplift on option premium.',
                'When tradable quotes are sparse, multiply spread by liquidity_penalty_multiplier.'
            ],
        }

        ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
        out_json = os.path.join(DATA_DIR, f'sol_option_calibrated_params_{ts}.json')
        out_latest = os.path.join(DATA_DIR, 'sol_option_calibrated_params_latest.json')
        with open(out_json, 'w', encoding='utf-8') as f:
            json.dump(calibrated, f, indent=2)
        with open(out_latest, 'w', encoding='utf-8') as f:
            json.dump(calibrated, f, indent=2)

        print('Calibrated option simulation parameters:')
        print(json.dumps(calibrated['option_pricing_params'], indent=2))
        print('Saved:', out_json)
        print('Saved:', out_latest)

Calibrated option simulation parameters:
{
  "iv_atm_anchor": 0.56496,
  "iv_floor": 0.55421,
  "iv_cap": 0.6306,
  "iv_skew_per_10pct_moneyness": -0.039396693954233945,
  "iv_term_slope_per_30d": -0.02943996267950178,
  "spread_pct_median": 0.0,
  "spread_pct_stress_p75": 0.0,
  "liquidity_penalty_multiplier": 1.1825688073394496,
  "iv_rv_median": null
}
Saved: data/sol_option_calibrated_params_20260414_091910.json
Saved: data/sol_option_calibrated_params_latest.json


## How to Extend Further

To add another venue, implement a connector that returns normalized rows matching `OPTION_COLS`, then register it in `CONNECTORS`.

Recommended additions (if public APIs are available):
- Aevo options (SOL if listed)
- Zeta Markets (Solana options)
- Any broker API you have credentials for (Derivatives desk / OTC RFQ logs)

For production pricing calibration, you should also add:
- Risk-free curve input (USD rates by tenor)
- Stablecoin basis / borrow curve
- Cross-venue de-duplication and stale quote filtering
- Contract-level liquidity screens (OI, volume, spread thresholds)